# Predict Tags on 5 Test Dataset Samples
### Comparing Multi-Class RF-DETR vs Old `model.plan` (TensorRT)

This notebook evaluates **5 sample images from the test dataset split** using:
1. **Multi-Class RF-DETR** (`.pth` checkpoint) — Fine-tuned category detection (`blue_aisle`, `blue_bay`, `location_tag`).
2. **Old Model** (`model.plan` TensorRT Engine + `config.pbtxt`) — Universal TensorRT execution with DBNet text detection & YOLO object detection decoders.

In [ ]:
# 1. Imports & Environment Setup
import os
import re
import json
import cv2
import torch
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from torchvision.ops import batched_nms

try:
    import tensorrt as trt
    HAS_TRT = True
except ImportError:
    HAS_TRT = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | TensorRT available: {HAS_TRT}")


In [ ]:
# 2. Load 5 Sample Images from the TEST Dataset Split
TEST_DIR = "multi_class_train_rfdetr/dataset_full_data/test"
if not os.path.exists(TEST_DIR):
    TEST_DIR = "multi_class_train_rfdetr/dataset_sample_1000/test"

IMAGES_DIR = "multi_class_train_rfdetr/images"
ANN_FILE = os.path.join(TEST_DIR, "_annotations.coco.json")

CLASSES = ["blue_aisle", "blue_bay", "location_tag"]
test_images = []

if os.path.exists(ANN_FILE):
    with open(ANN_FILE) as f:
        coco = json.load(f)
    CLASSES = [c["name"] for c in sorted(coco.get("categories", []), key=lambda x: x["id"])]
    test_images = [os.path.join(IMAGES_DIR, im["file_name"]) for im in coco.get("images", [])[:5]]
elif os.path.exists(TEST_DIR):
    test_images = [os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR) if f.lower().endswith((".jpg", ".png"))][:5]
else:
    test_images = [f"test_image_{i}.jpg" for i in range(1, 6)]

print(f"Categories ({len(CLASSES)}): {CLASSES}")
print("Selected 5 Test Images:")
for idx, path in enumerate(test_images, 1):
    print(f"  {idx}. {path} (exists: {os.path.exists(path)})")


In [ ]:
# 3. Load Models (Multi-Class RF-DETR & Full model.plan with config.pbtxt)

# --- 3A. Load Multi-Class RF-DETR ---
from rfdetr import RFDETRBase
from rfdetr.models.lwdetr import LWDETR

# Disable automatic background download of coco pretrain
RFDETRBase.maybe_download_pretrain_weights = lambda self: None
RFDETRBase.load_pretrain_weights = lambda self: None

# Safe weight loader to match layer tensor shapes
def _safe_load(self, state_dict, strict=True):
    cur = self.state_dict()
    filtered = {k.replace("model.", "").replace("module.", ""): v for k, v in state_dict.items()}
    filtered = {k: v for k, v in filtered.items() if k in cur and cur[k].shape == v.shape}
    return torch.nn.Module.load_state_dict(self, filtered, strict=False)
LWDETR.load_state_dict = _safe_load

rf_ckpt = "multi_class_train_rfdetr/model/best_model_full_data.pth"
if not os.path.exists(rf_ckpt):
    rf_ckpt = "multi_class_train_rfdetr/model/best_model_sample_1000.pth"

wrapper = RFDETRBase(num_classes=len(CLASSES), resolution=1008, pretrain_weights=None)
if hasattr(wrapper, "model") and hasattr(wrapper.model, "model") and hasattr(wrapper.model.model, "load_state_dict"):
    rfdetr = wrapper.model.model
elif hasattr(wrapper, "model") and hasattr(wrapper.model, "load_state_dict"):
    rfdetr = wrapper.model
else:
    rfdetr = wrapper

if os.path.exists(rf_ckpt):
    ckpt = torch.load(rf_ckpt, map_location="cpu", weights_only=False)
    rfdetr.load_state_dict(ckpt.get("model", ckpt), strict=False)
    print(f"Loaded RF-DETR checkpoint: {rf_ckpt}")
else:
    print(f"Notice: RF-DETR checkpoint not found at: {rf_ckpt}")
rfdetr.to(device).eval()

# --- 3B. Parse Triton config.pbtxt for Old Model ---
CONFIG_PBTXT_PATH = "config.pbtxt"
def parse_triton_config(pbtxt_path: str) -> dict:
    if not os.path.exists(pbtxt_path): return {}
    info = {'inputs': [], 'outputs': []}
    try:
        with open(pbtxt_path, 'r', encoding='utf-8') as f:
            c = f.read()
        in_match = re.search(r'input\s*[:\[\{](.*?)(?:output|$)', c, re.DOTALL)
        if in_match:
            section = in_match.group(1)
            name = re.search(r'name\s*:\s*"([^"]+)"', section)
            dims = re.search(r'dims\s*:\s*\[\s*([\d\s,-]+)\s*\]', section)
            if name and dims:
                info['inputs'].append({'name': name.group(1), 'dims': [int(x.strip()) for x in dims.group(1).split(',') if x.strip()]})
        out_match = re.search(r'output\s*[:\[\{](.*)', c, re.DOTALL)
        if out_match:
            section = out_match.group(1)
            name = re.search(r'name\s*:\s*"([^"]+)"', section)
            dims = re.search(r'dims\s*:\s*\[\s*([\d\s,-]+)\s*\]', section)
            if name and dims:
                info['outputs'].append({'name': name.group(1), 'dims': [int(x.strip()) for x in dims.group(1).split(',') if x.strip()]})
    except Exception as e:
        print(f"Error reading {pbtxt_path}: {e}")
    return info

trt_cfg = parse_triton_config(CONFIG_PBTXT_PATH)
plan_in_dims = trt_cfg['inputs'][0]['dims'] if trt_cfg.get('inputs') else [1, 3, 640, 640]
plan_in_h = plan_in_dims[-2] if plan_in_dims[-2] > 0 else 640
plan_in_w = plan_in_dims[-1] if plan_in_dims[-1] > 0 else 640
print(f"Config.pbtxt settings -> Resolution: {plan_in_w}x{plan_in_h}, Inputs: {trt_cfg.get('inputs')}, Outputs: {trt_cfg.get('outputs')}")

# --- 3C. Universal TensorRT Engine Runner (model.plan) ---
class TensorRTRunner:
    def __init__(self, plan_path: str, device: torch.device):
        self.plan_path = plan_path
        self.device = device
        self.is_ready = False
        self.inputs, self.outputs = [], []
        
        if not HAS_TRT:
            print("[TensorRT Runner] TensorRT Python library not installed.")
            return
        if not os.path.exists(plan_path):
            print(f"[TensorRT Runner] Plan file not found at: {plan_path}")
            return
            
        trt_logger = trt.Logger(trt.Logger.WARNING)
        with open(plan_path, "rb") as f, trt.Runtime(trt_logger) as runtime:
            self.engine = runtime.deserialize_cuda_engine(f.read())
        if self.engine is None:
            return
            
        self.context = self.engine.create_execution_context()
        self._inspect_io()
        self.is_ready = True
        print(f"[TensorRT Runner] Successfully loaded engine from: {plan_path}")

    def _inspect_io(self):
        if hasattr(self.engine, "num_io_tensors"):
            for i in range(self.engine.num_io_tensors):
                name = self.engine.get_tensor_name(i)
                mode = self.engine.get_tensor_mode(name)
                shape = list(self.engine.get_tensor_shape(name))
                meta = {"name": name, "shape": shape}
                (self.inputs if mode == trt.TensorIOMode.INPUT else self.outputs).append(meta)
        else:
            for i in range(self.engine.num_bindings):
                name = self.engine.get_binding_name(i)
                is_in = self.engine.binding_is_input(i)
                shape = list(self.engine.get_binding_shape(i))
                meta = {"name": name, "shape": shape}
                (self.inputs if is_in else self.outputs).append(meta)

    def infer(self, input_tensor: torch.Tensor):
        assert self.is_ready
        input_tensor = input_tensor.contiguous().to(self.device)
        output_tensors = []
        if hasattr(self.context, "set_tensor_address"):
            self.context.set_tensor_address(self.inputs[0]["name"], input_tensor.data_ptr())
            for out_m in self.outputs:
                shape = [input_tensor.shape[0] if s < 0 and idx == 0 else (abs(s) if s < 0 else s) for idx, s in enumerate(out_m["shape"])]
                out_t = torch.empty(shape, dtype=torch.float32, device=self.device)
                self.context.set_tensor_address(out_m["name"], out_t.data_ptr())
                output_tensors.append(out_t)
            self.context.execute_async_v3(torch.cuda.current_stream().cuda_stream)
            torch.cuda.synchronize()
        return output_tensors

PLAN_MODEL_PATH = "model.plan" if os.path.exists("model.plan") else "location_tag_text_det.engine"
trt_runner = TensorRTRunner(PLAN_MODEL_PATH, device)
if trt_runner.is_ready and trt_runner.inputs:
    plan_in_h = trt_runner.inputs[0]["shape"][-2] if trt_runner.inputs[0]["shape"][-2] > 0 else plan_in_h
    plan_in_w = trt_runner.inputs[0]["shape"][-1] if trt_runner.inputs[0]["shape"][-1] > 0 else plan_in_w


In [ ]:
# 4. Prediction Functions (Full Old Model Decoders + RF-DETR)
CONF_THRESH = 0.50
IOU_THRESH = 0.50

def box_cxcywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)

def predict_tags_rfdetr(img_path):
    """Predicts multi-class tags using RF-DETR."""
    if not os.path.exists(img_path):
        return []
    img = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img.size
    
    # Resize to 1008x1008 and normalize
    x = TF.to_tensor(img.resize((1008, 1008))).unsqueeze(0)
    x = TF.normalize(x, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]).to(device)
    
    with torch.no_grad():
        out = rfdetr(x)
        
    scores, classes = out["pred_logits"][0].sigmoid().max(-1)
    boxes = out["pred_boxes"][0]
    keep = scores > CONF_THRESH
    
    preds = []
    for s, c, (cx, cy, w, h) in zip(scores[keep], classes[keep], boxes[keep]):
        x1 = float(max(0, (cx - w / 2) * orig_w))
        y1 = float(max(0, (cy - h / 2) * orig_h))
        x2 = float(min(orig_w, (cx + w / 2) * orig_w))
        y2 = float(min(orig_h, (cy + h / 2) * orig_h))
        tag_name = CLASSES[int(c)] if int(c) < len(CLASSES) else f"class_{int(c)}"
        preds.append({"tag": tag_name, "score": round(float(s), 2), "box": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
    return preds


def predict_tags_plan(img_path):
    """Predicts tags using old model.plan with all decoders (DBNet heatmap & YOLO bounding boxes)."""
    if not trt_runner.is_ready or not os.path.exists(img_path):
        return []
        
    cv_img = cv2.imread(img_path)
    if cv_img is None:
        return []
    orig_h, orig_w = cv_img.shape[:2]
    
    # Preprocess image to engine input shape
    resized = cv2.resize(cv_img, (plan_in_w, plan_in_h))
    rgb = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    inp_trt = torch.from_numpy(rgb).permute(2, 0, 1).float().div(255.0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        raw_outs = trt_runner.infer(inp_trt)
    if not raw_outs:
        return []
        
    out = raw_outs[0]
    preds = []
    
    # Decoders for Old Model:
    # Decoder A: DBNet / Text Detection (Probability map [1, 1, H, W] or [1, H, W])
    if (out.ndim == 4 and out.shape[1] in (1, 2)) or (out.ndim == 3 and out.shape[0] in (1, 2) and out.shape[1] > 10):
        prob_map = (out[0, 0] if out.ndim == 4 else out[0]).sigmoid() if out.max() > 1.0 else (out[0, 0] if out.ndim == 4 else out[0])
        mask = (prob_map > 0.30).cpu().numpy().astype(np.uint8)
        contours, _ = cv2.findContours(mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            bx, by, bw, bh = cv2.boundingRect(cnt)
            if bw >= 4 and bh >= 4:
                sub_mask = mask[by:by+bh, bx:bx+bw]
                box_score = float(prob_map[by:by+bh, bx:bx+bw].mean().item()) if sub_mask.size > 0 else 0.60
                if box_score > CONF_THRESH:
                    x1 = float(bx * (orig_w / plan_in_w))
                    y1 = float(by * (orig_h / plan_in_h))
                    x2 = float((bx + bw) * (orig_w / plan_in_w))
                    y2 = float((by + bh) * (orig_h / plan_in_h))
                    preds.append({"tag": "location_tag", "score": round(box_score, 2), "box": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]})
                    
    # Decoder B: YOLO / Object Detection Format
    elif out.ndim in (2, 3):
        p = out[0] if out.ndim == 3 else out
        if p.shape[0] < p.shape[1] and p.shape[0] <= 100:
            p = p.transpose(0, 1)
        if p.shape[1] >= 5:
            scores_all = p[:, 4:].sigmoid() if (p[:, 4:].max() > 1.0 or p[:, 4:].min() < 0) else p[:, 4:]
            scs, lbs = scores_all.max(-1)
            keep = scs > CONF_THRESH
            if keep.sum() > 0:
                b_xyxy = box_cxcywh_to_xyxy(p[keep, :4])
                nms_k = batched_nms(b_xyxy, scs[keep], lbs[keep], IOU_THRESH)
                scaled = b_xyxy[nms_k] * torch.tensor([orig_w/plan_in_w, orig_h/plan_in_h, orig_w/plan_in_w, orig_h/plan_in_h], device=b_xyxy.device)
                for box, score, label in zip(scaled.cpu().tolist(), scs[keep][nms_k].cpu().tolist(), lbs[keep][nms_k].cpu().tolist()):
                    tag_name = CLASSES[int(label)] if int(label) < len(CLASSES) else "location_tag"
                    preds.append({"tag": tag_name, "score": round(float(score), 2), "box": [round(x, 1) for x in box]})
                    
    return preds


In [ ]:
# 5. Run Prediction on the 5 Test Images & Print Detected Tags
for i, path in enumerate(test_images, 1):
    print(f"\n==================== Test Image {i}: {os.path.basename(path)} ====================")
    if not os.path.exists(path):
        print(f"Image file not found: {path}")
        continue
        
    rf_tags = predict_tags_rfdetr(path)
    plan_tags = predict_tags_plan(path)
    
    print(f"Multi-Class RF-DETR ({len(rf_tags)} tags detected):")
    for t in rf_tags:
        print(f"   - {t['tag']:<15} (confidence: {t['score']:.2f}) -> Box: {t['box']}")
        
    print(f"Old Model (model.plan) ({len(plan_tags)} tags detected):")
    for t in plan_tags:
        print(f"   - {t['tag']:<15} (confidence: {t['score']:.2f}) -> Box: {t['box']}")


In [ ]:
# 6. Display Predicted Tags on the 5 Test Images Side-by-Side
COLORS = {"blue_aisle": "deepskyblue", "blue_bay": "orange", "location_tag": "lime"}

def draw_boxes(img_path, tags):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    for t in tags:
        color = COLORS.get(t["tag"], "red")
        draw.rectangle(t["box"], outline=color, width=3)
        draw.text((t["box"][0], max(0, t["box"][1] - 16)), f"{t['tag']} {t['score']}", fill=color)
    return im

valid_images = [p for p in test_images if os.path.exists(p)]
if valid_images:
    fig, axes = plt.subplots(len(valid_images), 2, figsize=(14, 4 * len(valid_images)))
    if len(valid_images) == 1:
        axes = np.array([axes])
        
    for i, p in enumerate(valid_images):
        # Left: Old model.plan
        axes[i, 0].imshow(draw_boxes(p, predict_tags_plan(p)))
        axes[i, 0].set_title(f"Test Image {i+1}: Old model.plan", fontsize=10)
        axes[i, 0].axis("off")
        
        # Right: Multi-Class RF-DETR
        axes[i, 1].imshow(draw_boxes(p, predict_tags_rfdetr(p)))
        axes[i, 1].set_title(f"Test Image {i+1}: Multi-Class RF-DETR", fontsize=10, fontweight="bold")
        axes[i, 1].axis("off")
        
    plt.tight_layout()
    plt.show()
else:
    print("Place your test images in the path above to view visual bounding boxes.")
